In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# CSV'yi oku ve ön işle
df = pd.read_csv("all_data.csv")
df["kategori"] = df["kategori"].replace("disci", "dis")

# Pivot işlemleri
df_nufus = df[df["veri_tipi"] == "nufus"][["il_adı", "yıl", "değer"]].rename(columns={"değer": "nufus"})
df_other = df[df["veri_tipi"] != "nufus"]
df_pivot = df_other.pivot_table(index=["il_adı", "yıl"], columns=["veri_tipi", "kategori"], values="değer", aggfunc="sum")
df_pivot.columns = ['_'.join(col).strip() for col in df_pivot.columns.values]
df_pivot = df_pivot.reset_index()

# Nüfusla birleştir, temizle ve bölge ekle
df_merged = pd.merge(df_pivot, df_nufus, on=["il_adı", "yıl"], how="left").dropna()
df_merged["bolge"] = df_merged["il_adı"].apply(bolge_bul)

# Hedef değişkenler
hedefler = [col for col in df_merged.columns if col not in ["il_adı", "yıl", "nufus", "bolge"]]
tahminler = []

# Tahmin döngüsü (LR + RF)
for il in df_merged["il_adı"].unique():
    df_il = df_merged[df_merged["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef in hedefler:
        df_target = df_il[["yıl", "nufus", hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        train = df_target[df_target["yıl"] <= 2021]
        test = df_target[df_target["yıl"] == 2023]
        if test.empty:
            continue

        X_train = train[["yıl", "nufus"]]
        y_train = train[hedef]
        X_test = test[["yıl", "nufus"]]
        y_test = test[hedef]

        # Linear Regression
        lr_model = LinearRegression().fit(X_train, y_train)
        y_pred_lr = lr_model.predict(X_test)

        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
        y_pred_rf = rf_model.predict(X_test)

        tahminler.append({
            "il_adı": il,
            "bolge": df_il["bolge"].iloc[0],
            "hedef": hedef,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": y_pred_lr[0],
            "tahmin_2023_rf": y_pred_rf[0]
        })

# Bölge bazında topla ve kaydet
df_tahmin = pd.DataFrame(tahminler)
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[["gercek_2023", "tahmin_2023_lr", "tahmin_2023_rf"]].sum().reset_index()
bolgesel.to_csv("bolgesel_2023_lr_rf_tahmin.csv", index=False)
